# ✍️ Phase 1 — Part 3: Auto-Transcription + Manual Correction

---

## What this notebook does
1. Uses a **baseline Whisper model** to auto-generate Bangla transcripts for every clip
2. Saves results as **CSV files** (one per dialect) in Google Drive
3. You then **manually correct** the CSV in Google Sheets/Excel

## What gets saved & where
| File | Location | Persists? |
|------|----------|----------|
| Auto-generated CSV | `dataset/dhaka_raw.csv` | ✅ Yes |
| Auto-generated CSV | `dataset/chittagong_raw.csv` | ✅ Yes |
| Script copy | `scripts/auto_transcribe.py` | ✅ Yes |

## If Colab disconnects mid-transcription?
The script has **resume support** — it checks which clips are already in the CSV and skips them. Just re-run the cell.

## After this notebook:
You MUST manually correct the CSV before training. Bad transcripts = bad model.

---

## Step 0: Mount Drive & Load Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os
config = json.load(open('/content/drive/MyDrive/NSU_499A/config.json'))
BASE = config['base_path']
print(f'✅ Project root: {BASE}')

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets torch torchaudio accelerate
print('✅ All packages installed')

## Step 2: Auto-Transcribe All Clips with Whisper

### How it works:
- Loads OpenAI's `whisper-small` model (pre-trained, no fine-tuning yet)
- Runs it on every `.wav` clip in `clips/{dialect}/`
- Saves the output as a CSV with columns: `file_path`, `transcript`, `dialect`, `verified`
- The `verified` column starts as `NO` — you change it to `YES` after manual correction

### Resume support:
If the CSV already exists, the script loads it and **only processes clips not yet transcribed**. This means if Colab disconnects after transcribing 200 clips, it will continue from clip 201.

### ⏱️ Estimated time:
- ~360 clips ≈ 15-25 minutes on T4 GPU

In [ ]:
# ════════════════════════════════════════════════════════════════
# AUTO-TRANSCRIPTION WITH RESUME SUPPORT
# ════════════════════════════════════════════════════════════════

from transformers import pipeline
import pandas as pd
import os, csv, time

# Load Whisper ASR pipeline
print('🔄 Loading Whisper-small model (this takes ~1 minute first time)...')
asr = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-small',
    device=0,  # Use GPU
    generate_kwargs={'language': 'bn', 'task': 'transcribe'}
)
print('✅ Model loaded!\n')

def transcribe_folder(clips_dir, dialect, output_csv):
    """Transcribe all WAV files in a folder. Supports resume."""
    
    # Get all clip files
    all_files = sorted([f for f in os.listdir(clips_dir) if f.endswith('.wav')])
    
    # Load existing CSV if it exists (resume support)
    existing_rows = []
    already_done = set()
    if os.path.exists(output_csv):
        df_existing = pd.read_csv(output_csv)
        existing_rows = df_existing.to_dict('records')
        already_done = set(df_existing['file_path'].tolist())
        print(f'  📄 Found existing CSV with {len(already_done)} entries. Resuming...')
    
    # Find clips not yet transcribed
    remaining = []
    for fname in all_files:
        full_path = os.path.join(clips_dir, fname)
        if full_path not in already_done:
            remaining.append(fname)
    
    if len(remaining) == 0:
        print(f'  ✅ All {len(all_files)} clips already transcribed!')
        return len(all_files)
    
    print(f'  🎙️ Transcribing {len(remaining)} new clips ({len(already_done)} already done)...')
    
    rows = existing_rows.copy()
    start_time = time.time()
    
    for i, fname in enumerate(remaining):
        full_path = os.path.join(clips_dir, fname)
        try:
            result = asr(full_path)
            rows.append({
                'file_path': full_path,
                'transcript': result['text'],
                'dialect': dialect,
                'verified': 'NO'
            })
        except Exception as e:
            print(f'    ❌ Error on {fname}: {e}')
            continue
        
        # Save progress every 50 clips (crash protection)
        if (i + 1) % 50 == 0 or (i + 1) == len(remaining):
            df = pd.DataFrame(rows)
            df.to_csv(output_csv, index=False, encoding='utf-8-sig')
            elapsed = time.time() - start_time
            print(f'    💾 Saved progress: {i+1}/{len(remaining)} done ({elapsed:.0f}s elapsed)')
    
    print(f'  ✅ Total: {len(rows)} transcriptions saved to {output_csv}')
    return len(rows)

# ── Run for both dialects ──
for dialect in ['dhaka', 'chittagong']:
    clips_dir = os.path.join(BASE, f'clips/{dialect}')
    output_csv = os.path.join(BASE, f'dataset/{dialect}_raw.csv')
    
    if not os.path.exists(clips_dir) or len(os.listdir(clips_dir)) == 0:
        print(f'\n⚠️ No clips found for {dialect}. Run Part 2 first.')
        continue
    
    print(f'\n📝 Processing {dialect}...')
    transcribe_folder(clips_dir, dialect, output_csv)

## Step 3: Preview the Auto-Generated Transcripts
Look at the first 10 rows to see what Whisper generated.

In [ ]:
for dialect in ['dhaka', 'chittagong']:
    csv_path = os.path.join(BASE, f'dataset/{dialect}_raw.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f'\n📋 {dialect}_raw.csv — {len(df)} rows')
        print(f'   Verified: {(df.verified == "YES").sum()} / {len(df)}')
        print(df[['file_path', 'transcript', 'verified']].head(10).to_string(index=False))
    else:
        print(f'\n⚠️ {dialect}_raw.csv not found')

## Step 4: Save Script Copy to Drive

In [ ]:
# Save a copy of the transcription script for reference
script_code = '''
# auto_transcribe.py — Auto-generate Bangla transcripts using Whisper
from transformers import pipeline
import os, csv

asr = pipeline('automatic-speech-recognition',
               model='openai/whisper-small',
               language='bn',
               generate_kwargs={'task': 'transcribe'})

def transcribe_folder(clips_dir, dialect, output_csv):
    rows = []
    files = sorted(f for f in os.listdir(clips_dir) if f.endswith('.wav'))
    for i, fname in enumerate(files):
        path = os.path.join(clips_dir, fname)
        result = asr(path)
        rows.append({
            'file_path': path,
            'transcript': result['text'],
            'dialect': dialect,
            'verified': 'NO'
        })
        if i % 20 == 0:
            print(f'Progress: {i}/{len(files)}')
    with open(output_csv, 'w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['file_path','transcript','dialect','verified'])
        w.writeheader(); w.writerows(rows)
    print(f'Saved {len(rows)} rows to {output_csv}')
'''
with open(os.path.join(BASE, 'scripts/auto_transcribe.py'), 'w') as f:
    f.write(script_code)
print('✅ Script saved to scripts/auto_transcribe.py')

---

## 🛑 STOP HERE — Manual Correction Required!

### What you MUST do before continuing:

1. **Open** `NSU_499A/dataset/dhaka_raw.csv` in **Google Sheets** or **Excel**
2. For each row:
   - **Listen** to the WAV file (column A has the path)
   - **Read** the auto-transcript in column B
   - **Fix** any errors — type the correct Bangla text
   - **Change** column D (`verified`) from `NO` to `YES`
   - **Delete** rows where audio is noisy, silent, or has background music
3. **Save** as CSV (UTF-8 encoding)
4. Repeat for `chittagong_raw.csv`

### ⚠️ DO NOT SKIP THIS STEP!
Auto-Whisper makes MANY mistakes on dialects. Bad transcripts = bad model.
Your entire project quality depends on transcript accuracy.

### Tips:
- Use **Google Sheets** — it auto-saves and supports UTF-8 Bangla well
- Open the CSV directly from Google Drive
- You can play WAV files using the Colab audio player (see cell below)

---

### (Helper) Play a specific clip to check its transcript

In [ ]:
# ════════════════════════════════════════════════════════════════
# AUDIO PLAYER — Listen to a specific clip
# Change clip_path to any WAV file path from the CSV
# ════════════════════════════════════════════════════════════════

from IPython.display import Audio, display

# Example: play the first clip
dialect = 'dhaka'  # or 'chittagong'
csv_path = os.path.join(BASE, f'dataset/{dialect}_raw.csv')

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    if len(df) > 0:
        clip_path = df.iloc[0]['file_path']
        transcript = df.iloc[0]['transcript']
        print(f'📂 File: {os.path.basename(clip_path)}')
        print(f'📝 Auto-transcript: {transcript}')
        print(f'🔊 Playing audio:')
        display(Audio(clip_path, rate=16000))
    else:
        print('CSV is empty')
else:
    print('CSV not found. Run transcription first.')

---
## ✅ Part 3 Complete!

**What is now saved in Google Drive:**
- `dataset/dhaka_raw.csv` — Auto-transcripts for all Dhaka clips
- `dataset/chittagong_raw.csv` — Auto-transcripts for all Chittagong clips
- `scripts/auto_transcribe.py` — Script backup

**What you need to do manually:**
- Correct transcripts in the CSV files
- Mark verified rows as `YES`

**After correction → Open `Phase1_Part4_Dataset_Split.ipynb`**

---